# Fase 4: Feedback Loop Real

## Objetivo
Cerrar el ciclo: **Recomendación → Ejecución → Resultado → Mejora**

### Componentes
1. **Tracking Table** — Qué se recomendó, qué se ejecutó, qué pasó
2. **Evaluación de Efectividad** — Conversión por acción/canal/timing/cluster
3. **A/B Testing Framework** — Grupo control vs tratamiento
4. **Drift Detection** — Cuándo reentrenar
5. **Señales de Reentrenamiento** — Triggers automáticos

### Flujo
```
Scoring mensual → Listas para marketing → Marketing ejecuta (parcial)
                                                    ↓
                                        Tracking: quién fue contactado,
                                        con qué acción, por qué canal
                                                    ↓
                                        3-6-12 meses después:
                                        ¿canjeó? ¿cuánto gastó?
                                                    ↓
                                        Evaluar: ¿la acción funcionó?
                                        ¿mejor que no hacer nada?
                                                    ↓
                                        Ajustar modelos y reglas
```

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Para visualizaciones
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_style("whitegrid")

BASE_DIR = Path.cwd().parent if Path.cwd().name == "fase4" else Path.cwd()
print(f"Base: {BASE_DIR}")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Esquema de Tracking — BigQuery SQL

La tabla central del feedback loop. Cada fila = una recomendación para un cliente en un mes.

In [ ]:
# ── SQL para crear la tabla de tracking en BigQuery ──────────────────────

TRACKING_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS `loyalty_analytics.action_tracking` (
  -- Identificadores
  tracking_id         STRING NOT NULL,     -- UUID
  scoring_month       DATE NOT NULL,       -- Mes del scoring (2025-03-01)
  cust_id             STRING NOT NULL,     -- Cliente
  
  -- Lo que el MODELO recomendó
  rec_prioridad       STRING,              -- Alta/Media/Baja/No contactar
  rec_expected_value  FLOAT64,             -- EV predicho
  rec_objetivo        STRING,              -- Activar/Acelerar/Empujar/etc.
  rec_accion          STRING,              -- Descuento/Educar/Experiencia/etc.
  rec_retailer        STRING,              -- Retailer recomendado
  rec_canal           STRING,              -- Digital/Email/Push/Presencial
  rec_timing          STRING,              -- Normal/Urgente/Inmediato
  rec_propensity      FLOAT64,             -- P(canje) predicha
  rec_uplift          FLOAT64,             -- Uplift predicho
  rec_cluster         STRING,              -- Cluster asignado
  rec_funnel_state    STRING,              -- Estado funnel en t0
  
  -- Lo que MARKETING ejecutó (se llena cuando marketing confirma)
  exec_flag           BOOL DEFAULT FALSE,  -- ¿Se ejecutó la acción?
  exec_date           DATE,                -- Fecha de ejecución
  exec_accion         STRING,              -- Acción real (puede diferir)
  exec_canal          STRING,              -- Canal real
  exec_retailer       STRING,              -- Retailer real
  exec_campaign_id    STRING,              -- ID de campaña (para cruzar)
  exec_cost           FLOAT64,             -- Costo de la acción ($CLP)
  
  -- Grupo de control
  control_flag        BOOL DEFAULT FALSE,  -- TRUE = holdout (no contactar)
  
  -- RESULTADOS observados (se llenan 1/3/6/12 meses después)
  -- Ventana 1 mes
  result_1m_redeemed    BOOL,              -- ¿Canjeó en 1 mes?
  result_1m_revenue     FLOAT64,           -- Revenue en 1 mes
  result_1m_redeem_amt  FLOAT64,           -- Monto canjeado en 1 mes
  
  -- Ventana 3 meses
  result_3m_redeemed    BOOL,
  result_3m_revenue     FLOAT64,
  result_3m_redeem_amt  FLOAT64,
  result_3m_n_canjes    INT64,             -- Número de canjes
  
  -- Ventana 6 meses
  result_6m_redeemed    BOOL,
  result_6m_revenue     FLOAT64,
  result_6m_redeem_amt  FLOAT64,
  result_6m_n_canjes    INT64,
  
  -- Ventana 12 meses (validación completa)
  result_12m_redeemed   BOOL,
  result_12m_revenue    FLOAT64,
  result_12m_redeem_amt FLOAT64,
  result_12m_n_canjes   INT64,
  result_12m_funnel     STRING,            -- Estado funnel 12m después
  
  -- Metadata
  created_at          TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  updated_at          TIMESTAMP
)
PARTITION BY scoring_month
CLUSTER BY cust_id, rec_prioridad;
"""

print("=== TABLA: action_tracking ===")
print("Columnas por sección:")
print(f"  Identificadores:      3")
print(f"  Recomendación modelo: 11")
print(f"  Ejecución marketing:  6")
print(f"  Control:              1")
print(f"  Resultados 1m:        3")
print(f"  Resultados 3m:        4")
print(f"  Resultados 6m:        4")
print(f"  Resultados 12m:       5")
print(f"  Metadata:             2")
print(f"  TOTAL:               39 columnas")
print()
print("Particionada por scoring_month, clusterizada por cust_id + prioridad")

## 2. A/B Testing Framework — Diseño de Grupo Control

Para medir causalidad real necesitamos un **holdout group**: clientes que el modelo recomienda contactar pero que NO se contactan.

**Regla**: 10% holdout aleatorio dentro de cada prioridad (estratificado).

In [ ]:
# ── A/B Split: Asignar grupo control estratificado ───────────────────────

def assign_ab_groups(df_scoring, holdout_pct=0.10, seed=42):
    """
    Asigna grupo control (holdout) estratificado por prioridad.
    
    - 10% holdout dentro de cada prioridad (Alta, Media, Baja)
    - "No contactar" se excluye del A/B (nunca se contactan)
    - Determinístico por cust_id (mismo cliente siempre en mismo grupo)
    
    Returns: df con columna 'ab_group' = 'treatment' | 'control' | 'excluded'
    """
    df = df_scoring.copy()
    df["ab_group"] = "treatment"
    
    # "No contactar" siempre excluido
    df.loc[df["prioridad"] == "No contactar", "ab_group"] = "excluded"
    
    # Holdout estratificado por prioridad
    contactable = df[df["ab_group"] == "treatment"].copy()
    
    for prioridad in contactable["prioridad"].unique():
        mask = contactable["prioridad"] == prioridad
        subset = contactable[mask]
        
        # Hash determinístico del cust_id para reproducibilidad
        hashes = subset["cust_id"].apply(lambda x: hash(x + str(seed)) % 100)
        control_mask = hashes < (holdout_pct * 100)
        
        # Marcar control en el df original
        control_ids = subset[control_mask]["cust_id"].values
        df.loc[df["cust_id"].isin(control_ids), "ab_group"] = "control"
    
    return df


# ── Simular con datos de scoring existente ──────────────────────────────

scoring_path = BASE_DIR / "fase3" / "output" / "scoring_202503.csv"
if scoring_path.exists():
    df_scoring = pd.read_csv(scoring_path)
    df_ab = assign_ab_groups(df_scoring, holdout_pct=0.10)
    
    print("=== A/B Group Assignment ===")
    print(f"\nDistribución general:")
    print(df_ab["ab_group"].value_counts().to_string())
    print(f"\nPor prioridad:")
    print(pd.crosstab(df_ab["prioridad"], df_ab["ab_group"], margins=True).to_string())
    print(f"\n% holdout por prioridad:")
    ct = pd.crosstab(df_ab["prioridad"], df_ab["ab_group"], normalize="index")
    if "control" in ct.columns:
        print(ct["control"].map(lambda x: f"{x:.1%}").to_string())
else:
    print("No se encontró scoring output. Ejecutar scoring_pipeline.py primero.")

## 3. Poblar Tracking Table — Insertar recomendaciones del scoring

Cada mes, después del scoring, se insertan las recomendaciones en `action_tracking`.

In [ ]:
# ── SQL: Insertar recomendaciones post-scoring ──────────────────────────

INSERT_RECOMMENDATIONS_SQL = """
-- Ejecutar mensualmente DESPUÉS del scoring
-- Inserta todas las recomendaciones + asigna grupo control

INSERT INTO `loyalty_analytics.action_tracking`
(tracking_id, scoring_month, cust_id,
 rec_prioridad, rec_expected_value, rec_objetivo, rec_accion,
 rec_retailer, rec_canal, rec_timing, rec_propensity, rec_uplift,
 rec_cluster, rec_funnel_state, control_flag)

SELECT
  GENERATE_UUID() AS tracking_id,
  s.scoring_month,
  s.cust_id,
  s.prioridad,
  s.expected_value,
  s.objetivo,
  s.accion,
  s.retailer_recomendado,
  s.canal,
  s.timing,
  s.propensity_score,
  s.uplift_x,
  s.cluster_name,
  s.funnel_state_at_t0,
  -- 10% holdout estratificado por prioridad
  (MOD(ABS(FARM_FINGERPRINT(CONCAT(s.cust_id, CAST(s.scoring_month AS STRING)))), 100) < 10
   AND s.prioridad != 'No contactar') AS control_flag
FROM `loyalty_analytics.scoring_monthly` s
WHERE s.scoring_month = @run_month
  AND s.cust_id NOT IN (
    SELECT cust_id FROM `loyalty_analytics.action_tracking`
    WHERE scoring_month = @run_month
  );
"""

print("=== INSERT RECOMMENDATIONS ===")
print("Se ejecuta mensualmente post-scoring.")
print("- Inserta ~500K filas (todos los clientes scored)")
print("- Asigna 10% holdout con FARM_FINGERPRINT (determinístico)")
print("- Idempotente: no inserta duplicados si ya existen")

## 4. Registrar Ejecución de Marketing

Cuando marketing ejecuta una campaña, actualiza `exec_flag` y los campos `exec_*`.  
Esto se puede hacer por upload CSV o cruce con la tabla de campañas existente (`evaluador_campanhas`).

In [ ]:
# ── SQL: Registrar ejecución desde tabla de campañas ────────────────────

REGISTER_EXECUTION_SQL = """
-- Cruzar con campañas ejecutadas del evaluador_campanhas
-- Se corre después de que marketing confirme la ejecución

UPDATE `loyalty_analytics.action_tracking` t
SET
  t.exec_flag = TRUE,
  t.exec_date = c.fecha_ejecucion,
  t.exec_accion = c.tipo_accion,
  t.exec_canal = c.canal,
  t.exec_retailer = c.retailer,
  t.exec_campaign_id = c.campaign_id,
  t.exec_cost = c.costo_unitario,
  t.updated_at = CURRENT_TIMESTAMP()
FROM `evaluador_campanhas.campanas_ejecutadas` c
WHERE t.cust_id = c.cust_id
  AND t.scoring_month = @run_month
  AND t.control_flag = FALSE     -- nunca marcar ejecución en control
  AND t.exec_flag = FALSE;       -- no sobrescribir
"""

# ── Alternativa: Upload CSV de marketing ────────────────────────────────

REGISTER_FROM_CSV_SQL = """
-- Si marketing sube un CSV con los clientes contactados:
-- 1. Subir CSV a tabla temporal `loyalty_analytics.exec_upload_temp`
-- 2. Cruzar:

UPDATE `loyalty_analytics.action_tracking` t
SET
  t.exec_flag = TRUE,
  t.exec_date = u.fecha,
  t.exec_accion = COALESCE(u.accion, t.rec_accion),
  t.exec_canal = COALESCE(u.canal, t.rec_canal),
  t.exec_campaign_id = u.campaign_id,
  t.updated_at = CURRENT_TIMESTAMP()
FROM `loyalty_analytics.exec_upload_temp` u
WHERE t.cust_id = u.cust_id
  AND t.scoring_month = @run_month
  AND t.control_flag = FALSE;
"""

print("=== REGISTRO DE EJECUCIÓN ===")
print("Dos opciones:")
print("  1. Cruce automático con evaluador_campanhas (si existe campaign_id)")
print("  2. Upload CSV manual de marketing")
print()
print("IMPORTANTE: El grupo control NUNCA se marca como ejecutado.")

## 5. Recoger Resultados — SQL de actualización progresiva

Se ejecuta mensualmente. Llena los campos `result_*` mirando transacciones y canjes reales.

In [ ]:
# ── SQL: Actualizar resultados observados ────────────────────────────────

UPDATE_RESULTS_SQL = """
-- Se ejecuta MENSUALMENTE
-- Actualiza ventanas de resultado según tiempo transcurrido desde scoring_month
-- Ejemplo: si hoy es junio 2025 y scoring_month = marzo 2025 → llena result_3m

DECLARE today DATE DEFAULT CURRENT_DATE();

-- ═══ Ventana 1 mes ═══
UPDATE `loyalty_analytics.action_tracking` t
SET
  t.result_1m_redeemed = COALESCE(r.redeemed, FALSE),
  t.result_1m_revenue = COALESCE(r.revenue, 0),
  t.result_1m_redeem_amt = COALESCE(r.redeem_amt, 0),
  t.updated_at = CURRENT_TIMESTAMP()
FROM (
  SELECT
    at.cust_id,
    at.scoring_month,
    COUNT(DISTINCT rd.redemption_id) > 0 AS redeemed,
    COALESCE(SUM(tx.tran_amt), 0) AS revenue,
    COALESCE(SUM(rd.points_redeemed), 0) AS redeem_amt
  FROM `loyalty_analytics.action_tracking` at
  LEFT JOIN `acc_loy_cl_prd.redemptions` rd
    ON at.cust_id = rd.cust_id
    AND rd.fecha >= at.scoring_month
    AND rd.fecha < DATE_ADD(at.scoring_month, INTERVAL 1 MONTH)
  LEFT JOIN `acc_loy_cl_prd.transactions` tx
    ON at.cust_id = tx.cust_id
    AND tx.fecha >= at.scoring_month
    AND tx.fecha < DATE_ADD(at.scoring_month, INTERVAL 1 MONTH)
  WHERE at.result_1m_redeemed IS NULL
    AND DATE_ADD(at.scoring_month, INTERVAL 1 MONTH) <= today
  GROUP BY at.cust_id, at.scoring_month
) r
WHERE t.cust_id = r.cust_id
  AND t.scoring_month = r.scoring_month;

-- ═══ Ventana 3 meses ═══
UPDATE `loyalty_analytics.action_tracking` t
SET
  t.result_3m_redeemed = COALESCE(r.redeemed, FALSE),
  t.result_3m_revenue = COALESCE(r.revenue, 0),
  t.result_3m_redeem_amt = COALESCE(r.redeem_amt, 0),
  t.result_3m_n_canjes = COALESCE(r.n_canjes, 0),
  t.updated_at = CURRENT_TIMESTAMP()
FROM (
  SELECT
    at.cust_id,
    at.scoring_month,
    COUNT(DISTINCT rd.redemption_id) > 0 AS redeemed,
    COALESCE(SUM(tx.tran_amt), 0) AS revenue,
    COALESCE(SUM(rd.points_redeemed), 0) AS redeem_amt,
    COUNT(DISTINCT rd.redemption_id) AS n_canjes
  FROM `loyalty_analytics.action_tracking` at
  LEFT JOIN `acc_loy_cl_prd.redemptions` rd
    ON at.cust_id = rd.cust_id
    AND rd.fecha >= at.scoring_month
    AND rd.fecha < DATE_ADD(at.scoring_month, INTERVAL 3 MONTH)
  LEFT JOIN `acc_loy_cl_prd.transactions` tx
    ON at.cust_id = tx.cust_id
    AND tx.fecha >= at.scoring_month
    AND tx.fecha < DATE_ADD(at.scoring_month, INTERVAL 3 MONTH)
  WHERE at.result_3m_redeemed IS NULL
    AND DATE_ADD(at.scoring_month, INTERVAL 3 MONTH) <= today
  GROUP BY at.cust_id, at.scoring_month
) r
WHERE t.cust_id = r.cust_id
  AND t.scoring_month = r.scoring_month;

-- ═══ Ventana 6 y 12 meses: misma lógica, cambiar INTERVAL ═══
-- (omitido por brevedad, misma estructura con INTERVAL 6/12 MONTH)
"""

print("=== UPDATE RESULTS ===")
print("Se ejecuta mensualmente. Solo actualiza ventanas cuyo periodo ya cerró.")
print()
print("Timeline para scoring_month = marzo 2025:")
print("  Abril 2025  → llena result_1m")
print("  Junio 2025  → llena result_3m")
print("  Sept 2025   → llena result_6m")
print("  Marzo 2026  → llena result_12m (validación completa)")

## 6. Evaluación de Efectividad — Treatment vs Control

La métrica clave: **¿los clientes contactados canjearon MÁS que los no contactados (control)?**

Si la diferencia no es significativa → la acción no está generando valor incremental.

In [ ]:
# ── SQL: Reporte de efectividad Treatment vs Control ─────────────────────

EFFECTIVENESS_SQL = """
-- Reporte de efectividad por ventana temporal
-- Ejecutar cuando result_3m esté disponible (3 meses post-scoring)

WITH metrics AS (
  SELECT
    scoring_month,
    
    -- Grupos
    CASE
      WHEN control_flag = TRUE THEN 'Control'
      WHEN exec_flag = TRUE THEN 'Tratamiento'
      ELSE 'No ejecutado'
    END AS grupo,
    
    rec_prioridad,
    rec_accion,
    rec_canal,
    rec_cluster,
    
    -- Métricas
    COUNT(*) AS n_clientes,
    
    -- Conversión 3m
    COUNTIF(result_3m_redeemed = TRUE) AS n_canjearon_3m,
    SAFE_DIVIDE(COUNTIF(result_3m_redeemed = TRUE), COUNT(*)) AS tasa_canje_3m,
    AVG(result_3m_revenue) AS avg_revenue_3m,
    SUM(result_3m_revenue) AS total_revenue_3m,
    AVG(result_3m_n_canjes) AS avg_canjes_3m,
    
    -- ROI
    SUM(result_3m_revenue) - SUM(COALESCE(exec_cost, 0)) AS profit_3m,
    SAFE_DIVIDE(
      SUM(result_3m_revenue) - SUM(COALESCE(exec_cost, 0)),
      SUM(COALESCE(exec_cost, 0))
    ) AS roi_3m
    
  FROM `loyalty_analytics.action_tracking`
  WHERE scoring_month = @eval_month
    AND result_3m_redeemed IS NOT NULL
  GROUP BY 1, 2, 3, 4, 5, 6
)

SELECT
  scoring_month,
  rec_prioridad,
  grupo,
  n_clientes,
  n_canjearon_3m,
  ROUND(tasa_canje_3m, 4) AS tasa_canje_3m,
  ROUND(avg_revenue_3m, 0) AS avg_revenue_3m,
  ROUND(total_revenue_3m, 0) AS total_revenue_3m,
  ROUND(roi_3m, 2) AS roi_3m
FROM metrics
ORDER BY rec_prioridad, grupo;
"""

print("=== REPORTE DE EFECTIVIDAD ===")
print()
print("Lo que quieres ver:")
print("┌─────────────┬──────────────┬───────────┬──────────────┬────────────┐")
print("│ Prioridad   │ Grupo        │ N clients │ Tasa canje   │ Avg Rev    │")
print("├─────────────┼──────────────┼───────────┼──────────────┼────────────┤")
print("│ Alta        │ Tratamiento  │   45,000  │    18.5%     │ $520,000   │")
print("│ Alta        │ Control      │    5,000  │    12.1%     │ $380,000   │")
print("│ Alta        │ INCREMENTAL  │           │    +6.4pp    │ +$140,000  │ ← ESTO ES EL VALOR")
print("├─────────────┼──────────────┼───────────┼──────────────┼────────────┤")
print("│ Media       │ Tratamiento  │   90,000  │     9.2%     │ $210,000   │")
print("│ Media       │ Control      │   10,000  │     7.8%     │ $190,000   │")
print("│ Media       │ INCREMENTAL  │           │    +1.4pp    │  +$20,000  │")
print("├─────────────┼──────────────┼───────────┼──────────────┼────────────┤")
print("│ Baja        │ Tratamiento  │  135,000  │     3.1%     │  $95,000   │")
print("│ Baja        │ Control      │   15,000  │     2.9%     │  $90,000   │")
print("│ Baja        │ INCREMENTAL  │           │    +0.2pp    │   +$5,000  │ ← no vale la pena")
print("└─────────────┴──────────────┴───────────┴──────────────┴────────────┘")
print()
print("Si Tratamiento ≈ Control → la acción no genera valor → cambiar estrategia")

In [ ]:
# ── Simulación local: Generar datos de feedback con mock ─────────────────

def simulate_feedback(df_ab, months_elapsed=3):
    """
    Simula resultados de feedback para demostrar el análisis.
    En producción esto viene de BigQuery (canjes y transacciones reales).
    """
    np.random.seed(42)
    df = df_ab.copy()
    
    # Simular ejecución: marketing ejecuta 70% de treatment
    treatment = df["ab_group"] == "treatment"
    exec_mask = treatment & (np.random.random(len(df)) < 0.70)
    df["exec_flag"] = exec_mask
    
    # Simular resultados basados en propensity + efecto del tratamiento
    prop = df["propensity_score"].fillna(0.1).values
    
    # Efecto del tratamiento: +5pp para ejecutados, +0 para control
    treatment_effect = np.where(df["exec_flag"], 0.05, 0)
    
    # Efecto adicional por calidad de match acción-cliente
    cluster_bonus = np.where(
        (df["cluster_name"] == "Cazadores de Canje") & (df["accion"].str.contains("Descuento", na=False)),
        0.03, 0  # +3pp si la acción matchea bien
    )
    
    # Probabilidad de canje = base + treatment + bonus
    p_canje = np.clip(prop + treatment_effect + cluster_bonus, 0, 1)
    
    # Generar resultado binario
    df[f"result_{months_elapsed}m_redeemed"] = np.random.random(len(df)) < p_canje
    
    # Revenue condicional al canje
    base_revenue = np.random.lognormal(mean=11, sigma=1.5, size=len(df))  # ~$60K-$500K
    df[f"result_{months_elapsed}m_revenue"] = np.where(
        df[f"result_{months_elapsed}m_redeemed"],
        base_revenue * (1 + treatment_effect * 2),  # Tratados gastan más
        base_revenue * 0.3  # No canjearon pero igual compran algo
    )
    
    df[f"result_{months_elapsed}m_n_canjes"] = np.where(
        df[f"result_{months_elapsed}m_redeemed"],
        np.random.poisson(1.5, len(df)),
        0
    )
    
    return df


if scoring_path.exists():
    df_feedback = simulate_feedback(df_ab, months_elapsed=3)
    
    # Crear columna de grupo para análisis
    df_feedback["grupo"] = np.where(
        df_feedback["ab_group"] == "control", "Control",
        np.where(df_feedback["exec_flag"], "Tratamiento", "No ejecutado")
    )
    
    print(f"=== Simulación de Feedback (3 meses) ===")
    print(f"Total clientes: {len(df_feedback):,}")
    print(f"  Tratamiento (ejecutado): {(df_feedback['grupo'] == 'Tratamiento').sum():,}")
    print(f"  Control (holdout):       {(df_feedback['grupo'] == 'Control').sum():,}")
    print(f"  No ejecutado:            {(df_feedback['grupo'] == 'No ejecutado').sum():,}")
    print(f"  Excluido:                {(df_feedback['ab_group'] == 'excluded').sum():,}")

## 7. Análisis de Efectividad — Treatment vs Control

In [ ]:
# ── Análisis Treatment vs Control ────────────────────────────────────────

from scipy import stats

def analyze_effectiveness(df, window="3m"):
    """Analiza efectividad del tratamiento vs control."""
    
    redeemed_col = f"result_{window}_redeemed"
    revenue_col = f"result_{window}_revenue"
    
    # Filtrar solo treatment y control (excluir no ejecutados y excluded)
    treated = df[df["grupo"] == "Tratamiento"]
    control = df[df["grupo"] == "Control"]
    
    print(f"{'='*70}")
    print(f"  EFECTIVIDAD DEL TRATAMIENTO — Ventana {window}")
    print(f"{'='*70}")
    
    # ── Global ──
    t_rate = treated[redeemed_col].mean()
    c_rate = control[redeemed_col].mean()
    t_rev = treated[revenue_col].mean()
    c_rev = control[revenue_col].mean()
    
    # Test estadístico de proporciones (z-test)
    n_t, n_c = len(treated), len(control)
    p_pooled = (treated[redeemed_col].sum() + control[redeemed_col].sum()) / (n_t + n_c)
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_t + 1/n_c))
    z_stat = (t_rate - c_rate) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    print(f"\n{'─'*70}")
    print(f"  GLOBAL")
    print(f"{'─'*70}")
    print(f"  {'Métrica':<25} {'Tratamiento':>12} {'Control':>12} {'Delta':>12} {'p-value':>10}")
    print(f"  {'─'*71}")
    print(f"  {'Tasa de canje':<25} {t_rate:>11.1%} {c_rate:>11.1%} {t_rate-c_rate:>+11.1%} {p_value:>10.4f}")
    print(f"  {'Revenue promedio':<25} {t_rev:>11,.0f} {c_rev:>11,.0f} {t_rev-c_rev:>+11,.0f}")
    print(f"  {'N clientes':<25} {n_t:>11,} {n_c:>11,}")
    
    sig = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.1 else "ns"
    print(f"\n  Significancia: {sig} (p={p_value:.4f})")
    
    # ── Por prioridad ──
    print(f"\n{'─'*70}")
    print(f"  POR PRIORIDAD")
    print(f"{'─'*70}")
    print(f"  {'Prioridad':<15} {'Trat. canje':>12} {'Ctrl. canje':>12} {'Lift (pp)':>10} {'Revenue Δ':>12}")
    print(f"  {'─'*61}")
    
    for prio in ["Alta", "Media", "Baja"]:
        t_p = treated[treated["prioridad"] == prio]
        c_p = control[control["prioridad"] == prio]
        if len(t_p) > 0 and len(c_p) > 0:
            tr = t_p[redeemed_col].mean()
            cr = c_p[redeemed_col].mean()
            t_r = t_p[revenue_col].mean()
            c_r = c_p[revenue_col].mean()
            print(f"  {prio:<15} {tr:>11.1%} {cr:>11.1%} {tr-cr:>+9.1%} {t_r-c_r:>+11,.0f}")
    
    # ── Por acción ──
    print(f"\n{'─'*70}")
    print(f"  POR ACCIÓN RECOMENDADA")
    print(f"{'─'*70}")
    print(f"  {'Acción':<35} {'Trat.':>8} {'Ctrl.':>8} {'Lift':>8} {'N':>6}")
    print(f"  {'─'*65}")
    
    accion_col = "accion" if "accion" in df.columns else "rec_accion"
    for accion in sorted(treated[accion_col].dropna().unique()):
        t_a = treated[treated[accion_col] == accion]
        c_a = control[control[accion_col] == accion]
        if len(t_a) >= 10 and len(c_a) >= 3:
            tr = t_a[redeemed_col].mean()
            cr = c_a[redeemed_col].mean()
            print(f"  {accion[:35]:<35} {tr:>7.1%} {cr:>7.1%} {tr-cr:>+7.1%} {len(t_a):>5}")
    
    # ── Por canal ──
    print(f"\n{'─'*70}")
    print(f"  POR CANAL")
    print(f"{'─'*70}")
    canal_col = "canal" if "canal" in df.columns else "rec_canal"
    for canal in sorted(treated[canal_col].dropna().unique()):
        t_c = treated[treated[canal_col] == canal]
        c_c = control[control[canal_col] == canal]
        if len(t_c) >= 10 and len(c_c) >= 3:
            tr = t_c[redeemed_col].mean()
            cr = c_c[redeemed_col].mean()
            print(f"  {canal:<20} Trat: {tr:.1%}  Ctrl: {cr:.1%}  Lift: {tr-cr:+.1%}  (N={len(t_c)})")
    
    return {
        "global_lift_pp": t_rate - c_rate,
        "global_p_value": p_value,
        "revenue_delta": t_rev - c_rev,
        "significant": p_value < 0.05
    }

if scoring_path.exists():
    results = analyze_effectiveness(df_feedback, "3m")

## 8. Visualizaciones de Feedback

In [ ]:
# ── Visualizaciones ──────────────────────────────────────────────────────

if scoring_path.exists():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    redeemed_col = "result_3m_redeemed"
    revenue_col = "result_3m_revenue"
    
    # Solo treatment y control
    df_tc = df_feedback[df_feedback["grupo"].isin(["Tratamiento", "Control"])].copy()
    
    # ── 1. Tasa de canje por grupo y prioridad ──
    ax = axes[0, 0]
    ct = df_tc.groupby(["prioridad", "grupo"])[redeemed_col].mean().unstack()
    ct = ct.reindex(["Alta", "Media", "Baja"])
    ct.plot(kind="bar", ax=ax, color=["#2196F3", "#FF9800"])
    ax.set_title("Tasa de Canje: Treatment vs Control", fontweight="bold")
    ax.set_ylabel("Tasa de canje")
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(title="Grupo")
    
    # ── 2. Revenue promedio por grupo ──
    ax = axes[0, 1]
    rev_by_group = df_tc.groupby(["prioridad", "grupo"])[revenue_col].mean().unstack()
    rev_by_group = rev_by_group.reindex(["Alta", "Media", "Baja"])
    rev_by_group.plot(kind="bar", ax=ax, color=["#2196F3", "#FF9800"])
    ax.set_title("Revenue Promedio: Treatment vs Control", fontweight="bold")
    ax.set_ylabel("Revenue promedio ($)")
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(title="Grupo")
    
    # ── 3. Lift incremental por acción ──
    ax = axes[1, 0]
    accion_col = "accion"
    lifts = []
    for accion in df_tc[accion_col].dropna().unique():
        t_a = df_tc[(df_tc[accion_col] == accion) & (df_tc["grupo"] == "Tratamiento")]
        c_a = df_tc[(df_tc[accion_col] == accion) & (df_tc["grupo"] == "Control")]
        if len(t_a) >= 10 and len(c_a) >= 3:
            lift = t_a[redeemed_col].mean() - c_a[redeemed_col].mean()
            lifts.append({"accion": accion[:25], "lift_pp": lift, "n": len(t_a)})
    
    if lifts:
        df_lifts = pd.DataFrame(lifts).sort_values("lift_pp", ascending=True)
        colors = ["#4CAF50" if x > 0 else "#F44336" for x in df_lifts["lift_pp"]]
        ax.barh(df_lifts["accion"], df_lifts["lift_pp"], color=colors)
        ax.set_title("Lift Incremental por Acción (pp)", fontweight="bold")
        ax.set_xlabel("Lift (puntos porcentuales)")
        ax.axvline(x=0, color="black", linewidth=0.5)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    
    # ── 4. Propensity predicha vs tasa real ──
    ax = axes[1, 1]
    df_tc["propensity_bin"] = pd.qcut(df_tc["propensity_score"].fillna(0), 10, duplicates="drop")
    cal = df_tc.groupby("propensity_bin").agg(
        predicted=("propensity_score", "mean"),
        actual=(redeemed_col, "mean")
    ).dropna()
    ax.scatter(cal["predicted"], cal["actual"], s=80, zorder=3)
    lims = [0, max(cal["predicted"].max(), cal["actual"].max()) * 1.1]
    ax.plot(lims, lims, "--", color="gray", alpha=0.5, label="Calibración perfecta")
    ax.set_title("Calibración: P(canje) predicha vs real", fontweight="bold")
    ax.set_xlabel("Propensity predicha")
    ax.set_ylabel("Tasa de canje real")
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(BASE_DIR / "fase4" / "feedback_effectiveness.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Guardado: fase4/feedback_effectiveness.png")

## 9. Drift Detection — ¿Cuándo reentrenar?

Monitorear 3 tipos de drift:
1. **Feature drift** (PSI) — Las distribuciones de las features cambian
2. **Prediction drift** — Los scores se mueven
3. **Performance drift** — El modelo predice peor (necesita resultados reales)

In [ ]:
# ── Population Stability Index (PSI) ─────────────────────────────────────

def compute_psi(expected, actual, bins=10):
    """
    Calcula PSI entre dos distribuciones.
    PSI < 0.1  → sin cambio significativo
    PSI 0.1-0.2 → cambio moderado (monitorear)
    PSI > 0.2  → cambio significativo (reentrenar)
    """
    # Crear bins basados en la distribución esperada
    breakpoints = np.quantile(expected.dropna(), np.linspace(0, 1, bins + 1))
    breakpoints = np.unique(breakpoints)
    
    expected_pct = np.histogram(expected.dropna(), bins=breakpoints)[0] / len(expected.dropna())
    actual_pct = np.histogram(actual.dropna(), bins=breakpoints)[0] / len(actual.dropna())
    
    # Evitar log(0)
    expected_pct = np.clip(expected_pct, 0.001, None)
    actual_pct = np.clip(actual_pct, 0.001, None)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


def drift_report(df_train, df_current, features):
    """Genera reporte de drift para features clave."""
    
    print(f"{'='*60}")
    print(f"  DRIFT REPORT — Population Stability Index")
    print(f"{'='*60}")
    print(f"  {'Feature':<30} {'PSI':>8} {'Estado':>15}")
    print(f"  {'─'*53}")
    
    alerts = {"green": 0, "yellow": 0, "red": 0}
    red_features = []
    
    for feat in features:
        if feat in df_train.columns and feat in df_current.columns:
            psi = compute_psi(df_train[feat], df_current[feat])
            
            if psi > 0.2:
                status = "REENTRENAR"
                alerts["red"] += 1
                red_features.append(feat)
            elif psi > 0.1:
                status = "Monitorear"
                alerts["yellow"] += 1
            else:
                status = "OK"
                alerts["green"] += 1
            
            print(f"  {feat:<30} {psi:>8.4f} {status:>15}")
    
    print(f"\n  Resumen: {alerts['green']} OK, {alerts['yellow']} monitorear, {alerts['red']} reentrenar")
    
    if alerts["red"] >= 3:
        print(f"\n  *** ALERTA ROJA: {alerts['red']} features con PSI > 0.2 → REENTRENAR MODELOS ***")
    
    return alerts, red_features


# ── Simular drift con datos mock ────────────────────────────────────────

if scoring_path.exists():
    # Simular "datos de entrenamiento" vs "datos actuales" con perturbación
    np.random.seed(123)
    df_current = df_scoring.copy()
    
    key_features = [
        "propensity_score", "uplift_x", "expected_value"
    ]
    # Agregar features numéricas disponibles
    numeric_cols = df_scoring.select_dtypes(include=[np.number]).columns.tolist()
    key_features = [f for f in numeric_cols if df_scoring[f].nunique() > 5][:12]
    
    # Simular drift: perturbar algunas features
    df_drifted = df_current.copy()
    if len(key_features) >= 3:
        # Shift en 2 features para simular drift
        for feat in key_features[:2]:
            df_drifted[feat] = df_drifted[feat] * 1.3 + np.random.normal(0, df_drifted[feat].std() * 0.2, len(df_drifted))
    
    alerts, red_features = drift_report(df_current, df_drifted, key_features)

## 10. Triggers de Reentrenamiento

Reglas automáticas para decidir cuándo reentrenar.

In [ ]:
# ── Lógica de decisión de reentrenamiento ────────────────────────────────

def should_retrain(drift_alerts, effectiveness_results, months_since_train=None):
    """
    Decide si hay que reentrenar basado en múltiples señales.
    
    Returns: (decision, reason, urgency)
    """
    reasons = []
    urgency = "none"
    
    # ── Trigger 1: Feature drift ──
    if drift_alerts["red"] >= 3:
        reasons.append(f"PSI > 0.2 en {drift_alerts['red']} features")
        urgency = "alta"
    elif drift_alerts["yellow"] >= 5:
        reasons.append(f"PSI 0.1-0.2 en {drift_alerts['yellow']} features")
        if urgency != "alta":
            urgency = "media"
    
    # ── Trigger 2: Performance degradation ──
    if effectiveness_results and effectiveness_results.get("significant") == False:
        reasons.append("Lift del tratamiento no es estadísticamente significativo")
        urgency = "alta"
    
    if effectiveness_results and effectiveness_results.get("global_lift_pp", 0) < 0.02:
        reasons.append(f"Lift global < 2pp ({effectiveness_results['global_lift_pp']:.1%})")
        if urgency != "alta":
            urgency = "media"
    
    # ── Trigger 3: Tiempo ──
    if months_since_train and months_since_train >= 12:
        reasons.append(f"Modelos tienen {months_since_train} meses sin reentrenar")
        if urgency == "none":
            urgency = "baja"
    elif months_since_train and months_since_train >= 6:
        reasons.append(f"Modelos tienen {months_since_train} meses — evaluar reentrenamiento")
        if urgency == "none":
            urgency = "baja"
    
    # ── Decisión ──
    if urgency == "alta":
        decision = "REENTRENAR AHORA"
    elif urgency == "media":
        decision = "PLANIFICAR REENTRENAMIENTO"
    elif urgency == "baja":
        decision = "MONITOREAR (reentrenar si empeora)"
    else:
        decision = "OK — No reentrenar"
    
    return decision, reasons, urgency


# ── Evaluar ──
decision, reasons, urgency = should_retrain(
    drift_alerts=alerts if scoring_path.exists() else {"red": 0, "yellow": 0, "green": 0},
    effectiveness_results=results if scoring_path.exists() else None,
    months_since_train=4  # Ejemplo: 4 meses desde entrenamiento
)

print(f"{'='*60}")
print(f"  DECISIÓN DE REENTRENAMIENTO")
print(f"{'='*60}")
print(f"\n  Decisión:  {decision}")
print(f"  Urgencia:  {urgency.upper()}")
print(f"\n  Razones:")
for r in reasons:
    print(f"    - {r}")
if not reasons:
    print(f"    - Sin alertas activas")

print(f"""
{'─'*60}
  REGLAS DE REENTRENAMIENTO
{'─'*60}
  Trigger                          │ Umbral        │ Acción
  ─────────────────────────────────┼───────────────┼──────────────
  PSI > 0.2 en 3+ features        │ ROJO          │ Reentrenar
  Lift T vs C no significativo     │ p > 0.05      │ Reentrenar
  Lift global < 2pp                │ AMARILLO      │ Planificar
  PSI 0.1-0.2 en 5+ features      │ AMARILLO      │ Planificar
  12+ meses sin reentrenar         │ BAJO          │ Planificar
  Tasa canje real < predicha 25%+  │ ROJO          │ Recalibrar
{'─'*60}""")

## 11. Ajuste Automático de Reglas del Decision Engine

El feedback loop más valioso: **ajustar las reglas del decision engine basado en qué funciona**.

Si "Descuento personalizado" no genera lift para "Exploradores" → cambiar la acción.  
Si "Digital" funciona mejor que "Email" para "Alta prioridad" → cambiar el canal.

In [ ]:
# ── Optimización de reglas basada en resultados ──────────────────────────

def optimize_decision_rules(df_feedback, window="3m"):
    """
    Analiza qué combinaciones cluster×acción×canal funcionan mejor
    y sugiere cambios a las reglas del decision engine.
    """
    redeemed_col = f"result_{window}_redeemed"
    revenue_col = f"result_{window}_revenue"
    
    # Solo tratamiento ejecutado con resultados
    executed = df_feedback[
        (df_feedback["grupo"] == "Tratamiento") & 
        (df_feedback[redeemed_col].notna())
    ].copy()
    
    control = df_feedback[
        (df_feedback["grupo"] == "Control") & 
        (df_feedback[redeemed_col].notna())
    ].copy()
    
    if len(executed) == 0 or len(control) == 0:
        print("Sin datos suficientes para optimizar.")
        return {}
    
    suggestions = []
    
    # ── Evaluar cada combinación cluster × acción ──
    print(f"{'='*70}")
    print(f"  OPTIMIZACIÓN DE REGLAS — basada en resultados {window}")
    print(f"{'='*70}")
    
    accion_col = "accion" if "accion" in executed.columns else "rec_accion"
    canal_col = "canal" if "canal" in executed.columns else "rec_canal"
    
    print(f"\n{'─'*70}")
    print(f"  EFECTIVIDAD POR CLUSTER × ACCIÓN")
    print(f"{'─'*70}")
    print(f"  {'Cluster':<20} {'Acción':<25} {'Conv.':>6} {'Lift vs Ctrl':>12} {'N':>5}")
    print(f"  {'─'*68}")
    
    for cluster in sorted(executed["cluster_name"].dropna().unique()):
        c_ctrl = control[control["cluster_name"] == cluster]
        ctrl_rate = c_ctrl[redeemed_col].mean() if len(c_ctrl) > 0 else 0
        
        for accion in sorted(executed[accion_col].dropna().unique()):
            subset = executed[(executed["cluster_name"] == cluster) & (executed[accion_col] == accion)]
            if len(subset) >= 5:
                conv_rate = subset[redeemed_col].mean()
                lift = conv_rate - ctrl_rate
                print(f"  {cluster[:20]:<20} {accion[:25]:<25} {conv_rate:>5.1%} {lift:>+11.1%} {len(subset):>5}")
                
                if lift < 0.01:
                    suggestions.append({
                        "cluster": cluster,
                        "current_accion": accion,
                        "issue": f"Lift bajo ({lift:+.1%})",
                        "suggestion": "Cambiar acción o canal"
                    })
    
    # ── Mejor canal por cluster ──
    print(f"\n{'─'*70}")
    print(f"  MEJOR CANAL POR CLUSTER")
    print(f"{'─'*70}")
    
    for cluster in sorted(executed["cluster_name"].dropna().unique()):
        canal_perf = executed[executed["cluster_name"] == cluster].groupby(canal_col)[redeemed_col].agg(["mean", "count"])
        canal_perf = canal_perf[canal_perf["count"] >= 3].sort_values("mean", ascending=False)
        if len(canal_perf) > 0:
            best = canal_perf.index[0]
            best_rate = canal_perf.iloc[0]["mean"]
            print(f"  {cluster[:25]:<25} → Mejor: {best:<12} ({best_rate:.1%} conversión)")
    
    # ── Sugerencias de cambio ──
    if suggestions:
        print(f"\n{'─'*70}")
        print(f"  SUGERENCIAS DE CAMBIO")
        print(f"{'─'*70}")
        for s in suggestions:
            print(f"  Cluster: {s['cluster']}")
            print(f"    Acción actual: {s['current_accion']}")
            print(f"    Problema: {s['issue']}")
            print(f"    Sugerencia: {s['suggestion']}")
            print()
    
    return suggestions

if scoring_path.exists():
    suggestions = optimize_decision_rules(df_feedback, "3m")

## 12. ROI del Sistema — Cuánto valor genera

La métrica final: **¿cuánto revenue incremental genera el sistema vs el costo de operarlo?**

In [ ]:
# ── ROI del sistema completo ──────────────────────────────────────────────

def compute_system_roi(df_feedback, window="3m", cost_per_action=500, 
                        infra_cost_monthly=2_000_000):
    """
    Calcula ROI del sistema de loyalty intelligence.
    
    Args:
        cost_per_action: Costo promedio por acción de marketing ($CLP)
        infra_cost_monthly: Costo mensual de infraestructura (BQ + Colab + dashboard)
    """
    redeemed_col = f"result_{window}_redeemed"
    revenue_col = f"result_{window}_revenue"
    
    treated = df_feedback[df_feedback["grupo"] == "Tratamiento"]
    control = df_feedback[df_feedback["grupo"] == "Control"]
    
    if len(treated) == 0 or len(control) == 0:
        print("Sin datos para calcular ROI.")
        return
    
    # Revenue incremental = (Revenue tratamiento - Revenue control) × N tratados
    rev_treatment = treated[revenue_col].mean()
    rev_control = control[revenue_col].mean()
    incremental_per_client = rev_treatment - rev_control
    n_treated = len(treated)
    
    total_incremental = incremental_per_client * n_treated
    
    # Costos
    action_costs = n_treated * cost_per_action
    total_costs = action_costs + infra_cost_monthly
    
    # ROI
    net_value = total_incremental - total_costs
    roi = net_value / total_costs if total_costs > 0 else 0
    
    # Escalar a 500K clientes (producción)
    scale_factor = 500_000 / max(n_treated, 1)
    
    print(f"{'='*60}")
    print(f"  ROI DEL SISTEMA — Ventana {window}")
    print(f"{'='*60}")
    print(f"""
  MOCK ({n_treated:,} clientes tratados)
  ─────────────────────────────────────
  Revenue incremental/cliente:  ${incremental_per_client:>12,.0f}
  Revenue incremental total:    ${total_incremental:>12,.0f}
  Costo acciones marketing:     ${action_costs:>12,.0f}
  Costo infraestructura:        ${infra_cost_monthly:>12,.0f}
  ─────────────────────────────────────
  Valor neto:                   ${net_value:>12,.0f}
  ROI:                          {roi:>12.1%}
  
  PROYECCIÓN PRODUCCIÓN (~350K tratados de 500K)
  ─────────────────────────────────────
  Revenue incremental total:    ${incremental_per_client * 350_000:>12,.0f}
  Costo acciones marketing:     ${350_000 * cost_per_action:>12,.0f}
  Costo infraestructura:        ${infra_cost_monthly:>12,.0f}
  ─────────────────────────────────────
  Valor neto proyectado:        ${incremental_per_client * 350_000 - 350_000 * cost_per_action - infra_cost_monthly:>12,.0f}
  ROI proyectado:               {(incremental_per_client * 350_000 - 350_000 * cost_per_action - infra_cost_monthly) / (350_000 * cost_per_action + infra_cost_monthly):>12.1%}
""")
    
    # Breakdown por prioridad
    print(f"  POR PRIORIDAD")
    print(f"  {'Prioridad':<12} {'N':>7} {'Rev Incr/cl':>14} {'Total Incr':>14} {'% del total':>12}")
    print(f"  {'─'*59}")
    
    for prio in ["Alta", "Media", "Baja"]:
        t_p = treated[treated["prioridad"] == prio]
        c_p = control[control["prioridad"] == prio]
        if len(t_p) > 0 and len(c_p) > 0:
            incr = t_p[revenue_col].mean() - c_p[revenue_col].mean()
            total = incr * len(t_p)
            pct = total / total_incremental if total_incremental > 0 else 0
            print(f"  {prio:<12} {len(t_p):>7,} ${incr:>13,.0f} ${total:>13,.0f} {pct:>11.1%}")

if scoring_path.exists():
    compute_system_roi(df_feedback, "3m")

## 13. Pipeline Completo en GitLab — Integración del Feedback Loop

Así queda el pipeline mensual completo con feedback loop integrado.

In [ ]:
# ── GitLab CI/CD completo con feedback loop ──────────────────────────────

GITLAB_CI_YAML = """
# .gitlab-ci.yml — Pipeline mensual Loyalty Intelligence
# Schedule: día 2 de cada mes, 6am

stages:
  - features        # Día 2: Calcular features del nuevo t0
  - scoring         # Día 2: Scoring Python (modelos)
  - tracking        # Día 2: Insertar recomendaciones + A/B split
  - feedback        # Día 2: Actualizar resultados de meses anteriores
  - monitoring      # Día 2: Drift detection + alertas
  - optimize        # Trimestral: Optimizar reglas del decision engine

# ── Stage 1: Features ──
run_features:
  stage: features
  image: google/cloud-sdk:latest
  script:
    - echo $GCP_SA_KEY > /tmp/sa.json
    - gcloud auth activate-service-account --key-file=/tmp/sa.json
    - bq query --use_legacy_sql=false --project_id=$GCP_PROJECT
        < loyalty_analytics/02_snapshots_features.sql

# ── Stage 2: Scoring ──
run_scoring:
  stage: scoring
  needs: [run_features]
  image: python:3.11
  script:
    - pip install google-cloud-bigquery xgboost scikit-learn pandas pyarrow
    - echo $GCP_SA_KEY > /tmp/sa.json
    - export GOOGLE_APPLICATION_CREDENTIALS=/tmp/sa.json
    - python loyalty_analytics/scoring_pipeline.py
        --project $GCP_PROJECT
        --month $(date -d 'last month' +%Y-%m)

# ── Stage 3: Tracking (insertar recomendaciones) ──
insert_tracking:
  stage: tracking
  needs: [run_scoring]
  image: google/cloud-sdk:latest
  script:
    - echo $GCP_SA_KEY > /tmp/sa.json
    - gcloud auth activate-service-account --key-file=/tmp/sa.json
    - bq query --use_legacy_sql=false --project_id=$GCP_PROJECT
        --parameter='run_month:DATE:$(date -d "last month" +%Y-%m-01)'
        < loyalty_analytics/04_insert_tracking.sql

# ── Stage 4: Feedback (actualizar resultados de meses anteriores) ──
update_results:
  stage: feedback
  needs: [insert_tracking]
  image: google/cloud-sdk:latest
  script:
    - echo $GCP_SA_KEY > /tmp/sa.json
    - gcloud auth activate-service-account --key-file=/tmp/sa.json
    - bq query --use_legacy_sql=false --project_id=$GCP_PROJECT
        < loyalty_analytics/05_update_results.sql

# ── Stage 5: Monitoring ──
run_monitoring:
  stage: monitoring
  needs: [update_results]
  image: python:3.11
  script:
    - pip install google-cloud-bigquery pandas scipy
    - echo $GCP_SA_KEY > /tmp/sa.json
    - export GOOGLE_APPLICATION_CREDENTIALS=/tmp/sa.json
    - python loyalty_analytics/monitoring.py
        --project $GCP_PROJECT
        --month $(date -d 'last month' +%Y-%m)
  artifacts:
    paths:
      - monitoring_report.json
    expire_in: 90 days
  allow_failure: true  # No bloquear pipeline si monitoring falla

# ── Stage 6: Optimización trimestral ──
optimize_rules:
  stage: optimize
  image: python:3.11
  script:
    - pip install google-cloud-bigquery pandas scipy
    - echo $GCP_SA_KEY > /tmp/sa.json
    - export GOOGLE_APPLICATION_CREDENTIALS=/tmp/sa.json
    - python loyalty_analytics/optimize_rules.py
        --project $GCP_PROJECT
  rules:
    - if: $QUARTERLY_RUN == "true"  # Manual trigger trimestral
"""

print("=== PIPELINE GITLAB COMPLETO ===")
print()
print("Flujo mensual automático (día 2 de cada mes):")
print()
print("  Stage 1: features      → SQL: calcular features nuevo t0")
print("  Stage 2: scoring       → Python: modelos → scoring 500K clientes")
print("  Stage 3: tracking      → SQL: insertar recomendaciones + A/B split")
print("  Stage 4: feedback      → SQL: actualizar resultados meses anteriores")
print("  Stage 5: monitoring    → Python: PSI + drift + alertas")
print("  Stage 6: optimize      → Python: optimizar reglas (trimestral)")
print()
print("Archivos en el repo:")
print("  loyalty_analytics/")
print("  ├── 01_exclusiones_muestra.sql")
print("  ├── 02_snapshots_features.sql")
print("  ├── 03_monitoring.sql")
print("  ├── 04_insert_tracking.sql")
print("  ├── 05_update_results.sql")
print("  ├── scoring_pipeline.py")
print("  ├── monitoring.py")
print("  ├── optimize_rules.py")
print("  └── .gitlab-ci.yml")

## 14. Resumen — El Ciclo Completo

```
MES 1: Scoring                    MES 2: Ejecución + resultado 1m
┌──────────────────────┐          ┌──────────────────────────────┐
│ Features (BQ SQL)    │          │ Marketing ejecuta acciones   │
│ → Scoring (Python)   │          │ → Registra en tracking table │
│ → Decision Engine    │          │ → Resultado 1m se actualiza  │
│ → Listas + A/B split │          │ → Drift check               │
│ → Tracking table     │          │                              │
└──────────────────────┘          └──────────────────────────────┘

MES 4: Resultado 3m               MES 7: Resultado 6m
┌──────────────────────┐          ┌──────────────────────────────┐
│ Actualizar result_3m │          │ Actualizar result_6m         │
│ → Treatment vs Ctrl  │          │ → Evaluación intermedia      │
│ → Lift significativo?│          │ → ¿Reentrenar?               │
│ → ROI parcial        │          │ → Optimizar reglas           │
└──────────────────────┘          └──────────────────────────────┘

MES 13: Validación completa
┌──────────────────────────────────────────────────────┐
│ Actualizar result_12m                                │
│ → Validación completa predicho vs real               │
│ → ROI definitivo del sistema                         │
│ → Decisión: mantener / reentrenar / rediseñar        │
│ → Feedback a reglas del decision engine               │
│ → Nuevo ciclo con modelos actualizados               │
└──────────────────────────────────────────────────────┘
```

### Esto es lo que lleva el proyecto de Mid-Senior a Staff:
1. **A/B Testing integrado** — grupo control estratificado
2. **Feedback loop real** — medir si las acciones funcionaron
3. **Optimización de reglas** — ajustar decision engine con datos reales
4. **ROI cuantificado** — valor incremental vs costo
5. **Triggers de reentrenamiento** — cuándo los modelos se degradan